In [1]:
!pip install yfinance

In [2]:
# Capstone Project: Stock Market Anomaly Detection
# Student: Burhanuddin Udaipurwala
# Username: iitg_ds_25011649

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("CAPSTONE: Stock Market Anomaly Detection\n")

# 1. FETCH DATA

def fetch_data(tickers):
    """Download stock data from Yahoo Finance"""
    data = {}
    print("Fetching data...")
    for t in tickers:
        df = yf.download(t, start='2018-01-01', end='2020-04-01', progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if 'Adj Close' not in df.columns:
            df['Adj Close'] = df['Close']
        data[t] = df[['Open','High','Low','Close','Adj Close','Volume']]
        print(f"  {t}: {len(df)} days")
    return data

# 2. ENGINEER FEATURES

def create_features(df, ticker):
    """
    Create 3 features (no leakage - using only past data):
    - ret_z: Return z-score (63-day window)
    - volz: Volume z-score (21-day window)
    - range_pct: Range percentile (63-day window)
    """
    df = df.copy()

    # Returns and log volume
    df['Return'] = df['Adj Close'].pct_change()
    df['LogVol'] = np.log(df['Volume'].replace(0, 1))

    # Feature 1: Return Z-Score (past 63 days)
    ret_mean = df['Return'].shift(1).rolling(63).mean()
    ret_std = df['Return'].shift(1).rolling(63).std()
    df['ret_z'] = (df['Return'] - ret_mean) / ret_std

    # Feature 2: Volume Z-Score (past 21 days)
    vol_mean = df['LogVol'].shift(1).rolling(21).mean()
    vol_std = df['LogVol'].shift(1).rolling(21).std()
    df['volz'] = (df['LogVol'] - vol_mean) / vol_std

    # Feature 3: Range Percentile (past 63 days)
    rng = (df['High'] - df['Low']) / df['Close']
    df['range_pct'] = rng.rolling(63).rank(pct=True) * 100

    df['Ticker'] = ticker
    df['Price'] = df['Adj Close']
    return df.dropna()

# 3. DETECT ANOMALIES (Rule-Based + Clustering)

def detect_anomalies(train, val, test):
    """
    Primary: Rule-based (|ret_z|>2.5, volz>2.5, range_pct>95)
    Secondary: K-Means + DBSCAN for validation
    """

    # Rule-based detection (simple thresholds)
    for df in [train, val, test]:
        df['anomaly_flag'] = (
            (np.abs(df['ret_z']) > 2.5) |
            (df['volz'] > 2.5) |
            (df['range_pct'] > 95)
        ).astype(int)

    # Type labeling
    def label_type(row):
        if not row['anomaly_flag']:
            return "Normal"
        types = []
        if np.abs(row['ret_z']) > 2.5:
            types.append("crash" if row['Return'] < 0 else "spike")
        if row['volz'] > 2.5:
            types.append("volume_shock")
        return "+".join(types) if types else "anomaly"

    def label_why(row):
        if not row['anomaly_flag']:
            return ""
        reasons = []
        if np.abs(row['ret_z']) > 2.5:
            reasons.append(f"|ret_z|>{abs(row['ret_z']):.1f}")
        if row['volz'] > 2.5:
            reasons.append(f"volz>{row['volz']:.1f}")
        if row['range_pct'] > 95:
            reasons.append(f"range>{row['range_pct']:.0f}")
        return "; ".join(reasons)

    test['type'] = test.apply(label_type, axis=1)
    test['why'] = test.apply(label_why, axis=1)

    # Clustering (for comparison)
    cols = ['ret_z', 'volz', 'range_pct']
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[cols])
    X_test = scaler.transform(test[cols])

    km = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_train)
    sil = silhouette_score(X_train, km.labels_)

    print(f"\nRule-based anomalies: {test['anomaly_flag'].sum()} ({test['anomaly_flag'].sum()/len(test)*100:.1f}%)")
    print(f"K-Means Silhouette: {sil:.3f}\n")

    return test, sil

# 4. MARKET METRICS

def compute_market_metrics(df):
    """Aggregate to daily market-level"""
    market = df.groupby(level=0).agg({
        'Return': 'mean',
        'anomaly_flag': 'mean'
    }).rename(columns={'Return': 'market_ret', 'anomaly_flag': 'flag_rate'})

    market['breadth'] = df.groupby(level=0).apply(lambda x: (x['Return'] > 0).sum() / len(x))
    market['market_anomaly_flag'] = (
        (market['market_ret'].abs() > market['market_ret'].abs().quantile(0.95)) |
        (market['breadth'] < 0.3)
    ).astype(int)

    return market

# 5. RUN PIPELINE

print("\n--- RUNNING PIPELINE ---\n")

# Fetch
tickers = ['QQQ', 'AAPL', 'MSFT', 'NVDA', 'AMZN', 'META']
data = fetch_data(tickers)

# Features
print("\nEngineering features...")
all_df = pd.concat([create_features(data[t], t) for t in tickers]).sort_index()
print(f"Combined: {len(all_df):,} stock-days")

# Split
train = all_df.loc['2018'].copy()
val = all_df.loc['2019'].copy()
test = all_df.loc['2020-01-01':'2020-03-31'].copy()
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

# Detect
test, sil_score = detect_anomalies(train, val, test)

# Market metrics
market = compute_market_metrics(test)

# 6. EXPORT CSVs

print("\n--- EXPORTING RESULTS ---\n")

# Stock anomalies
stock_csv = test[test['anomaly_flag']==1].reset_index()[
    ['Date','Ticker','anomaly_flag','type','Return','ret_z','volz','range_pct','why']
]
stock_csv.columns = ['date','ticker','anomaly_flag','type','ret','ret_z','volz','range_pct','why']
stock_csv.to_csv('daily_anomaly_card.csv', index=False)
print(f"✓ daily_anomaly_card.csv ({len(stock_csv)} anomalies)")

# Market metrics
market_csv = market.reset_index()[['Date','market_ret','breadth','market_anomaly_flag']]
market_csv.columns = ['date','market_ret','breadth','market_anomaly_flag']
market_csv.to_csv('market_day_table.csv', index=False)
print(f"✓ market_day_table.csv ({len(market_csv)} days)\n")

try:
    files.download('daily_anomaly_card.csv')
    files.download('market_day_table.csv')
except:
    pass

# 7. QUERY FUNCTIONS

def date_query(date):
    """Check anomalies for a specific date"""
    print(f"\nDATE QUERY: {date}\n")
    try:
        m = market.loc[date]
        print(f"Market: {'🚨 ANOMALY' if m['market_anomaly_flag'] else '✅ Normal'}")
        print(f"  Return: {m['market_ret']*100:.2f}% | Breadth: {m['breadth']*100:.0f}%\n")

        anoms = test.loc[date][test.loc[date]['anomaly_flag']==1]
        if not anoms.empty:
            print(f"Flagged Stocks ({len(anoms)}):")
            for _, r in anoms.iterrows():
                print(f"  {r['Ticker']:6s} | {r['type']:20s} | {r['Return']*100:5.2f}% | {r['why']}")
        else:
            print("No stock anomalies")
    except:
        print("Date not found")
    print()

def monthly_report(year, month):
    """Monthly summary table"""
    print(f"\nMONTHLY REPORT: {year}-{month:02d}\n")

    month_data = test[(test.index.year==year) & (test.index.month==month)]
    anoms = month_data[month_data['anomaly_flag']==1]

    print(f"Total Anomalies: {len(anoms)} ({len(anoms)/len(month_data)*100:.1f}%)\n")
    print(f"{'Date':<12} {'Ticker':<8} {'Type':<20} {'ret_z':<8} {'volz':<8} {'Why'}")
    print("-"*80)

    for date, r in anoms.nlargest(10, 'ret_z').iterrows():
        print(f"{str(date.date()):<12} {r['Ticker']:<8} {r['type']:<20} {r['ret_z']:>6.2f}  {r['volz']:>6.2f}  {r['why'][:30]}")
    print()

def dashboard(ticker):
    """Visual dashboard"""
    stock = test[test['Ticker']==ticker]
    anoms = stock[stock['anomaly_flag']==1]

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        row_heights=[0.5,0.25,0.25],
                        subplot_titles=(f'{ticker} Price','Z-Scores','Market Breadth'))

    fig.add_trace(go.Scatter(x=stock.index, y=stock['Price'], name='Price',
                            line=dict(color='blue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=anoms.index, y=anoms['Price'], mode='markers',
                            name='Anomaly', marker=dict(color='red',size=10,symbol='x')), row=1, col=1)

    fig.add_trace(go.Scatter(x=stock.index, y=stock['ret_z'], name='ret_z',
                            line=dict(color='green')), row=2, col=1)
    fig.add_trace(go.Scatter(x=stock.index, y=stock['volz'], name='volz',
                            line=dict(color='orange')), row=2, col=1)
    fig.add_hline(y=2.5, line_dash="dash", line_color="red", row=2, col=1)
    fig.add_hline(y=-2.5, line_dash="dash", line_color="red", row=2, col=1)

    fig.add_trace(go.Scatter(x=market.index, y=market['breadth'], name='Breadth',
                            fill='tozeroy', line=dict(color='purple')), row=3, col=1)
    fig.add_hline(y=0.3, line_dash="dash", line_color="red", row=3, col=1)

    fig.update_layout(height=900, title=f'{ticker} Anomaly Detection',
                     template='plotly_white', hovermode='x unified')
    fig.show()

# 8. DEMO

print("\n--- DEMO QUERIES ---\n")

date_query('2020-02-27')  # COVID crash
monthly_report(2020, 2)   # February 2020
dashboard('AAPL')          # Visual

print(f"\n")
print("✅ PROJECT COMPLETE")
print(f"\n")
print(f"\nAnomalies: {test['anomaly_flag'].sum()} ({test['anomaly_flag'].sum()/len(test)*100:.1f}%)")
print(f"Silhouette Score: {sil_score:.3f}\n")

CAPSTONE: Stock Market Anomaly Detection


--- RUNNING PIPELINE ---

Fetching data...
  QQQ: 565 days
  AAPL: 565 days
  MSFT: 565 days
  NVDA: 565 days
  AMZN: 565 days
  META: 565 days

Engineering features...
Combined: 3,006 stock-days
Train: 1,122 | Val: 1,512 | Test: 372

Rule-based anomalies: 126 (33.9%)
K-Means Silhouette: 0.296


--- EXPORTING RESULTS ---

✓ daily_anomaly_card.csv (126 anomalies)
✓ market_day_table.csv (62 days)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- DEMO QUERIES ---


DATE QUERY: 2020-02-27

Market: 🚨 ANOMALY
  Return: -5.46% | Breadth: 0%

Flagged Stocks (6):
  QQQ    | crash+volume_shock   | -5.01% | |ret_z|>5.0; volz>2.6; range>100
  AAPL   | crash+volume_shock   | -6.54% | |ret_z|>4.1; volz>2.7; range>97
  MSFT   | crash+volume_shock   | -7.05% | |ret_z|>5.3; volz>2.9; range>98
  AMZN   | crash                | -4.81% | |ret_z|>3.2; range>100
  NVDA   | anomaly              | -5.57% | range>98
  META   | anomaly              | -3.78% | range>95


MONTHLY REPORT: 2020-02

Total Anomalies: 50 (43.9%)

Date         Ticker   Type                 ret_z    volz     Why
--------------------------------------------------------------------------------
2020-02-14   NVDA     spike+volume_shock     3.59    3.62  |ret_z|>3.6; volz>3.6
2020-02-04   MSFT     spike                  3.24    1.35  |ret_z|>3.2; range>100
2020-02-04   QQQ      spike                  3.01    0.54  |ret_z|>3.0
2020-02-19   NVDA     spike                  2.74 



✅ PROJECT COMPLETE



Anomalies: 126 (33.9%)
Silhouette Score: 0.296

